# 01 · EDA dan Preprocessing Teks
**Mini Project NLP Part 1 · Kelompok 2**
Studi kasus: *Klasifikasi Masukan Pengguna terhadap Pengembangan Aplikasi Digital*

Notebook ini membaca `raw_dataset.csv` hasil notebook 00 (ulasan Google Play DANA, GoPay, OVO, ShopeePay yang sudah berlabel), melakukan **Exploratory Data Analysis (EDA)**, lalu mengubah teks ulasan yang informal menjadi `clean_text` melalui tahap preprocessing bertahap.

| Langkah | Isi |
|---|---|
| 1 | Konfigurasi (mode run otomatis, path, label, warna) |
| 2 | Import library |
| 3 | Memuat dataset dan pemeriksaan awal |
| 4 | Membuang ulasan hampir kembar (hanya beda huruf besar, tanda baca, atau emoji) |
| 5 | EDA: distribusi label, aplikasi, rating, panjang ulasan, ciri noise, kata terbanyak sebelum preprocessing |
| 6 | Preprocessing bertahap: case folding, regex cleaning, normalisasi slang, tokenisasi, stopword removal, stemming |
| 7 | Ringkasan preprocessing: tabel before/after, tabel token, statistik |
| 8 | EDA setelah preprocessing: kata terbanyak dan word cloud per label |
| 9 | Menyimpan `clean_dataset_part1.csv` dan file pendukung laporan |

**Lima label:** Permintaan Fitur, Bug Aplikasi, Desain Antarmuka, Performa Aplikasi, Keamanan Aplikasi.

Setiap tahap preprocessing disimpan di kolom tersendiri (`case_folded`, `cleaned`, `normalized`, `tokens`, `tokens_nostop`, `tokens_stemmed`, `clean_text`), sehingga perubahan dari satu tahap ke tahap berikutnya dapat ditunjukkan di laporan dan presentasi.

## 1. Konfigurasi

Notebook ini **tidak perlu diubah kodenya** antara *Run All* dan *Save Version*. Mode run dibaca otomatis dari variabel lingkungan Kaggle `KAGGLE_KERNEL_RUN_TYPE`:

| Nilai | Kapan | Perilaku |
|---|---|---|
| `Interactive` | Run All di editor notebook (juga saat dijalankan lokal) | **Smoke test**: hanya `SMOKE_SAMPLE_SIZE` ulasan (sampel acak berstrata per label), untuk memastikan semua sel berjalan |
| `Batch` | Save Version (Save & Run All) | **Full run**: seluruh ulasan diproses, hasilnya menjadi output resmi |

Input dicari otomatis: file `raw_dataset.csv` dicari secara rekursif di bawah `/kaggle/input`, sehingga nama folder dataset tidak berpengaruh. Tambahkan dataset `app-feedback-ewallet-reviews` lewat **Add Input** sebelum menjalankan notebook.

`LABEL_COLORS` menetapkan satu warna untuk setiap label. Warna yang sama dipakai di semua notebook agar grafik mudah dibandingkan.

In [ ]:
import os, glob

# ---- mode run otomatis: 'Interactive' (Run All di editor) | 'Batch' (Save Version) ----
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive')
SMOKE_TEST = RUN_TYPE != 'Batch'

INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
FIG_DIR = f'{OUTPUT_DIR}/figures'
RAW_FILE = 'raw_dataset.csv'
CLEAN_FILE = 'clean_dataset_part1.csv'

SMOKE_SAMPLE_SIZE = 3000                              # smoke test: sampel kecil berstrata per label
SAMPLE_SIZE = SMOKE_SAMPLE_SIZE if SMOKE_TEST else None  # None = semua baris

RANDOM_STATE = 42
N_EXAMPLES = 5          # jumlah contoh ulasan di tabel before/after
TOP_N_WORDS = 15        # jumlah kata terbanyak per label

LABELS = ['Permintaan Fitur', 'Bug Aplikasi', 'Desain Antarmuka', 'Performa Aplikasi', 'Keamanan Aplikasi']
APPS = ['DANA', 'GoPay', 'OVO', 'ShopeePay']
LABEL_COLORS = {
    'Permintaan Fitur': '#0072B2',
    'Bug Aplikasi': '#D55E00',
    'Desain Antarmuka': '#CC79A7',
    'Performa Aplikasi': '#E69F00',
    'Keamanan Aplikasi': '#009E73',
}

print(f'RUN_TYPE : {RUN_TYPE} -> {"SMOKE TEST" if SMOKE_TEST else "FULL RUN"}')
print(f'Data     : {"sampel " + str(SAMPLE_SIZE) + " ulasan" if SAMPLE_SIZE else "semua ulasan"}')

## 2. Import library

| Library | Kegunaan |
|---|---|
| `re` | Regular expression untuk membersihkan teks |
| `pandas`, `numpy` | Mengolah tabel data |
| `matplotlib` | Membuat grafik EDA |
| `wordcloud` | Membuat word cloud per label |
| `Sastrawi` | Stopword Bahasa Indonesia dan stemming (algoritma Nazief dan Adriani) |

`wordcloud` sudah tersedia di Kaggle, sedangkan `Sastrawi` belum, sehingga dipasang dengan `pip` jika belum ada. Pemasangan membutuhkan **Internet: On** di Settings Kaggle.

In [ ]:
import re, time, shutil, warnings
from collections import Counter
from multiprocessing import get_context

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import Sastrawi  # noqa: F401
except ImportError:
    !pip install -q Sastrawi
try:
    import wordcloud  # noqa: F401
except ImportError:
    !pip install -q wordcloud

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    from wordcloud import WordCloud
except ImportError as e:
    raise RuntimeError('Sastrawi atau wordcloud gagal dipasang. Aktifkan Internet (Settings, Internet: On), '
                       'lalu jalankan ulang notebook.') from e

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
os.makedirs(FIG_DIR, exist_ok=True)
# pola regex dengan grup (misalnya huruf berulang) hanya dipakai untuk mendeteksi, bukan mengambil grup
warnings.filterwarnings('ignore', message='This pattern is interpreted as a regular expression')


def save_fig(fig: plt.Figure, name: str) -> None:
    """Simpan grafik ke OUTPUT_DIR/figures dengan resolusi 200 dpi untuk laporan dan slide."""
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=200, bbox_inches='tight')

## 3. Memuat dataset

`raw_dataset.csv` dicari dengan `glob` secara rekursif (`**`) di bawah `/kaggle/input`. Jika tidak ditemukan, notebook berhenti dengan pesan yang menjelaskan Input yang harus ditambahkan.

Kolom dataset:

| Kolom | Isi |
|---|---|
| `text` | Teks ulasan asli dari Google Play |
| `label` | Salah satu dari lima label hasil pelabelan notebook 00 |
| `app` | Nama aplikasi (DANA, GoPay, OVO, ShopeePay) |
| `rating` | Bintang 1 sampai 5 yang diberikan pengulas |
| `date` | Tanggal ulasan |
| `source` | Urutan pengambilan di Google Play: `newest` (terbaru) atau `relevant` (paling relevan) |

File asli juga disalin ke Output agar Output notebook ini lengkap untuk folder pengumpulan (`raw_dataset.csv` dan `clean_dataset_part1.csv`).

In [ ]:
raw_paths = sorted(glob.glob(f'{INPUT_DIR}/**/{RAW_FILE}', recursive=True))
if not raw_paths:
    raise FileNotFoundError(f'{RAW_FILE} tidak ditemukan di {INPUT_DIR}. Tambahkan dataset '
                            '"app-feedback-ewallet-reviews" (output notebook 00) lewat Add Input.')
RAW_PATH = raw_paths[0]
print('Membaca:', RAW_PATH)

df_raw = pd.read_csv(RAW_PATH)
shutil.copy(RAW_PATH, f'{OUTPUT_DIR}/{RAW_FILE}')

print('Ukuran :', df_raw.shape)
df_raw.head()

### 3.1 Pemeriksaan awal

Sebelum analisis, dicek tipe kolom, nilai kosong, duplikat persis, dan apakah semua label sesuai dengan daftar `LABELS`. Label di luar daftar menandakan kesalahan di notebook sebelumnya.

In [ ]:
check = pd.DataFrame({
    'tipe': df_raw.dtypes.astype(str),
    'nilai_kosong': df_raw.isna().sum(),
    'nilai_unik': df_raw.nunique(),
})
display(check)

unknown_labels = set(df_raw['label']) - set(LABELS)
print('Duplikat persis pada text :', df_raw['text'].duplicated().sum())
print('Label di luar LABELS      :', unknown_labels or 'tidak ada')
assert not unknown_labels, 'Ada label yang tidak dikenal, periksa notebook 00.'

## 4. Membuang ulasan hampir kembar

Notebook 00 sudah membuang duplikat persis. Namun masih ada ulasan yang **isinya sama tetapi berbeda huruf besar, tanda baca, spasi, atau emoji**, misalnya `Loading nya lama banget` dan `Loading nya lama banget!!`. Ulasan seperti ini hampir pasti ditulis ulang oleh orang yang sama atau merupakan salinan.

Jika dibiarkan, satu salinan bisa masuk data latih dan salinan lain masuk data uji, sehingga akurasi model di notebook 03 terlihat lebih tinggi dari seharusnya.

Caranya: dibuat **kunci pembanding** `dedup_key` berupa teks huruf kecil yang hanya menyisakan huruf dan angka (pola `[^a-z0-9]+` diganti satu spasi). Ulasan dengan kunci yang sama dianggap kembar, dan hanya kemunculan pertama yang dipertahankan. Kunci ini hanya alat bantu dan tidak disimpan di output.

In [ ]:
def make_dedup_key(text: str) -> str:
    """Kunci pembanding: huruf kecil, hanya huruf dan angka, spasi tunggal."""
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()

df_raw['dedup_key'] = df_raw['text'].map(make_dedup_key)
dup_mask = df_raw['dedup_key'].duplicated(keep=False)
dup_groups = df_raw[dup_mask].sort_values('dedup_key')

n_groups = dup_groups['dedup_key'].nunique()
n_removed = df_raw['dedup_key'].duplicated().sum()
n_conflict = (dup_groups.groupby('dedup_key')['label'].nunique() > 1).sum()
print(f'Kelompok ulasan hampir kembar : {n_groups}')
print(f'Baris yang dibuang            : {n_removed}')
print(f'Kelompok dengan label berbeda : {n_conflict}')

dup_groups[['text', 'label', 'app']].head(10)

Hasil di atas memperlihatkan pasangan ulasan yang dibuang. Selanjutnya hanya kemunculan pertama dari setiap kelompok yang dipertahankan.

Pada mode smoke test, data diperkecil menjadi `SMOKE_SAMPLE_SIZE` baris dengan **sampel berstrata** (proporsi tiap label sama dengan data penuh) dan `random_state=RANDOM_STATE` agar hasilnya selalu sama. Pada Save Version seluruh data dipakai.

In [ ]:
df = (df_raw[~df_raw['dedup_key'].duplicated()]
      .drop(columns='dedup_key')
      .reset_index(drop=True))
print(f'Setelah dedup : {len(df):,} baris (dibuang {len(df_raw) - len(df)})')

if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    frac = SAMPLE_SIZE / len(df)
    df = (df.groupby('label', group_keys=False)
            .sample(frac=frac, random_state=RANDOM_STATE)
            .reset_index(drop=True))
    print(f'Smoke test    : sampel berstrata {len(df):,} baris')

## 5. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami data sebelum diolah: seberapa seimbang label, dari aplikasi mana ulasan berasal, bagaimana hubungan label dengan rating, seberapa panjang ulasan, dan noise apa saja yang perlu dibersihkan.

### 5.1 Distribusi label

Distribusi label menunjukkan apakah data seimbang. Ketidakseimbangan penting diketahui sejak awal karena memengaruhi pemilihan metrik dan strategi pelatihan di notebook 03 (misalnya `class_weight='balanced'` dan F1 macro).

In [ ]:
label_counts = df['label'].value_counts().reindex(LABELS)
label_table = pd.DataFrame({
    'jumlah': label_counts,
    'persen': (label_counts / len(df) * 100).round(2),
}).sort_values('jumlah', ascending=False)
label_table.loc['Total'] = [label_table['jumlah'].sum(), 100.0]
label_table['jumlah'] = label_table['jumlah'].astype(int)
display(label_table)

order = label_counts.sort_values().index
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(order, label_counts[order], color=[LABEL_COLORS[l] for l in order])
ax.bar_label(bars, labels=[f'{v:,} ({v / len(df) * 100:.1f}%)' for v in label_counts[order]], padding=4)
ax.set_title('Distribusi label ulasan')
ax.set_xlabel('Jumlah ulasan')
ax.set_xlim(0, label_counts.max() * 1.25)
plt.tight_layout()
save_fig(fig, '01_label_distribution')
plt.show()

print(f'Rasio label terbanyak : tersedikit = {label_counts.max() / label_counts.min():.1f} : 1')

### 5.2 Distribusi aplikasi per label

Tabel silang (crosstab) aplikasi terhadap label menunjukkan apakah suatu label hanya berasal dari satu aplikasi. Jika iya, model berisiko mempelajari nama aplikasi alih-alih isi keluhan.

In [ ]:
app_label = pd.crosstab(df['app'], df['label']).reindex(index=APPS, columns=LABELS)
app_label['Total'] = app_label.sum(axis=1)
display(app_label)

app_label_pct = pd.crosstab(df['app'], df['label'], normalize='index').reindex(index=APPS, columns=LABELS) * 100
fig, ax = plt.subplots(figsize=(9, 4))
app_label_pct.plot(kind='barh', stacked=True, ax=ax, color=[LABEL_COLORS[l] for l in LABELS], width=0.7)
ax.set_title('Komposisi label pada setiap aplikasi')
ax.set_xlabel('Persentase ulasan (%)')
ax.set_ylabel('')
ax.invert_yaxis()
ax.legend(title='Label', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.tight_layout()
save_fig(fig, '02_app_label_composition')
plt.show()

### 5.3 Rating per label

Rating (bintang) membantu memeriksa kewajaran label. Keluhan bug, performa, dan keamanan wajar didominasi bintang rendah, sedangkan permintaan fitur dan komentar desain lebih sering datang dari pengguna yang cukup puas (bintang menengah sampai tinggi).

In [ ]:
rating_label = (pd.crosstab(df['label'], df['rating'], normalize='index') * 100).reindex(LABELS).round(1)
rating_label['rata_rata_rating'] = df.groupby('label')['rating'].mean().reindex(LABELS).round(2)
display(rating_label)

rating_colors = ['#B2182B', '#EF8A62', '#BDBDBD', '#67A9CF', '#2166AC']
fig, ax = plt.subplots(figsize=(9, 4))
rating_label[[1, 2, 3, 4, 5]].plot(kind='barh', stacked=True, ax=ax, color=rating_colors, width=0.7)
ax.set_title('Sebaran rating pada setiap label')
ax.set_xlabel('Persentase ulasan (%)')
ax.set_ylabel('')
ax.invert_yaxis()
ax.legend(title='Bintang', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.tight_layout()
save_fig(fig, '03_rating_per_label')
plt.show()

### 5.4 Panjang ulasan per label

Panjang ulasan dihitung dalam jumlah kata (dipisah spasi) dan jumlah karakter. Informasi ini berguna untuk:
- mengetahui apakah suatu label cenderung ditulis lebih panjang,
- menentukan panjang maksimum input pada model deep learning di Part 2.

Boxplot menampilkan median (garis hitam), kuartil 1 sampai 3 (kotak), dan sebaran data (garis whisker). Pencilan disembunyikan agar kotak terlihat jelas.

In [ ]:
df['n_words_raw'] = df['text'].str.split().str.len()
df['n_chars_raw'] = df['text'].str.len()

length_table = (df.groupby('label')['n_words_raw']
                  .describe()[['mean', '50%', 'min', 'max']]
                  .rename(columns={'mean': 'rata_rata_kata', '50%': 'median_kata', 'min': 'min_kata', 'max': 'maks_kata'})
                  .reindex(LABELS).round(1))
length_table['rata_rata_karakter'] = df.groupby('label')['n_chars_raw'].mean().reindex(LABELS).round(1)
display(length_table)

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot([df.loc[df['label'] == l, 'n_words_raw'] for l in LABELS],
                showfliers=False, patch_artist=True, widths=0.6)
for patch, l in zip(bp['boxes'], LABELS):
    patch.set_facecolor(LABEL_COLORS[l])
    patch.set_alpha(0.75)
for median in bp['medians']:
    median.set_color('black')
ax.set_xticks(range(1, len(LABELS) + 1), LABELS)
ax.set_title('Panjang ulasan (jumlah kata) per label')
ax.set_ylabel('Jumlah kata')
plt.tight_layout()
save_fig(fig, '04_review_length_per_label')
plt.show()

### 5.5 Ciri noise pada teks mentah

Bagian ini menghitung berapa persen ulasan yang memuat noise khas ulasan aplikasi. Angka ini menjadi alasan mengapa setiap tahap preprocessing di bagian 6 diperlukan.

| Ciri | Cara mendeteksi |
|---|---|
| Huruf kapital | ada huruf `A-Z` |
| URL | pola `https?://` atau `www.` |
| Angka | ada digit `0-9` |
| Emoji | karakter di rentang Unicode emoji |
| Tanda baca berulang | dua atau lebih tanda `!` atau `?` berturut-turut |
| Huruf berulang | satu huruf muncul tiga kali atau lebih berturut-turut (`lemooot`) |
| Kata slang | memuat minimal satu kata dari kamus `SLANG` (bagian 6.3), misalnya `gk`, `bgt`, `apk` |

Kamus `SLANG` didefinisikan di bagian 6.3, sehingga baris terakhir dihitung setelah kamus tersedia (lihat 6.3).

In [ ]:
# rentang Unicode emoji ditulis sebagai escape agar tidak ada karakter emoji literal di kode
EMOJI_PATTERN = ('[\U0001F300-\U0001FAFF'   # simbol, wajah, gestur, objek
                 '\U00002600-\U000027BF'    # simbol lain-lain dan dingbat
                 '\U0001F1E6-\U0001F1FF'    # bendera
                 '\U0001F000-\U0001F2FF'    # mahjong, kartu, simbol berlingkar
                 '\u2B00-\u2BFF'            # panah dan bintang
                 '\u200D\uFE0F]+')          # penyambung emoji dan variation selector

noise_checks = {
    'Huruf kapital': r'[A-Z]',
    'URL': r'https?://|www\.',
    'Angka': r'\d',
    'Emoji': EMOJI_PATTERN,
    'Tanda baca berulang': r'[!?]{2,}',
    'Huruf berulang (3x atau lebih)': r'([A-Za-z])\1{2,}',
}
noise_table = pd.DataFrame({
    'jumlah_ulasan': {k: int(df['text'].str.contains(p, regex=True).sum()) for k, p in noise_checks.items()},
})
noise_table['persen'] = (noise_table['jumlah_ulasan'] / len(df) * 100).round(2)
noise_table

### 5.6 Contoh ulasan per label

Satu contoh acak untuk setiap label (dengan `random_state` tetap), untuk memberi gambaran isi ulasan sebelum dibersihkan.

In [ ]:
examples_per_label = (df.groupby('label', group_keys=False)
                        .sample(n=1, random_state=RANDOM_STATE)
                        .set_index('label')
                        .reindex(LABELS)[['app', 'rating', 'text']])
examples_per_label

### 5.7 Kata terbanyak sebelum preprocessing

Kata dihitung dari teks yang hanya diubah ke huruf kecil dan dipecah per spasi, tanpa pembersihan lain. Hasilnya didominasi kata umum seperti `di`, `saya`, `yang`, dan bentuk slang seperti `gak`, `ga`, `yg`, serta varian ejaan seperti `eror` dan `error`. Kata-kata ini tidak membedakan label, sehingga perlu stopword removal dan normalisasi slang. Kata terbanyak **per label** ditampilkan di bagian 8 setelah preprocessing, karena di sana hasilnya lebih bermakna.

In [ ]:
raw_word_counts = Counter(w for t in df['text'].str.lower() for w in t.split())
top_raw = pd.DataFrame(raw_word_counts.most_common(20), columns=['kata', 'frekuensi'])
print(f'Jumlah kata unik (hanya huruf kecil dan split spasi): {len(raw_word_counts):,}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_raw['kata'][::-1], top_raw['frekuensi'][::-1], color='#7F7F7F')
ax.set_title('20 kata terbanyak sebelum preprocessing (semua label)')
ax.set_xlabel('Frekuensi')
plt.tight_layout()
save_fig(fig, '05_top_words_before_preprocessing')
plt.show()

## 6. Preprocessing bertahap

Urutan tahap dan kolom hasilnya:

| No | Tahap | Kolom hasil | Contoh |
|---|---|---|---|
| 1 | Case folding | `case_folded` | `Apk LEMOT` menjadi `apk lemot` |
| 2 | Regex cleaning | `cleaned` | `lemooot!!! 100rb` menjadi `lemot` |
| 3 | Normalisasi slang | `normalized` | `gk bisa` menjadi `tidak bisa` |
| 4 | Tokenisasi | `tokens` | `tidak bisa login` menjadi `['tidak', 'bisa', 'login']` |
| 5 | Stopword removal | `tokens_nostop` | `['saya', 'tidak', 'bisa', 'login']` menjadi `['tidak', 'bisa', 'login']` |
| 6 | Stemming | `tokens_stemmed` | `['pembayaran', 'terpotong']` menjadi `['bayar', 'potong']` |
| 7 | Penggabungan | `clean_text` | token digabung kembali dengan spasi |

Urutan ini disengaja:
- **Case folding sebelum regex**, karena pola regex hanya menulis huruf kecil `a-z`.
- **Normalisasi slang sebelum stopword removal**, agar `gk` menjadi `tidak` terlebih dahulu, lalu `tidak` dipertahankan oleh daftar pengecualian stopword.
- **Stemming paling akhir**, karena stemming paling lambat. Setelah stopword dibuang, jumlah kata yang harus di-stem sudah lebih sedikit.

### 6.1 Case folding

Case folding mengubah semua huruf menjadi huruf kecil dengan `str.lower()`. Tanpa tahap ini, `Error`, `ERROR`, dan `error` dianggap tiga kata berbeda oleh BoW dan TF-IDF.

In [ ]:
df['case_folded'] = df['text'].str.lower()
df[['text', 'case_folded']].head(N_EXAMPLES)

### 6.2 Regex cleaning

**Regular expression (regex)** adalah pola untuk mencari bagian teks. Fungsi `re.sub(pola, pengganti, teks)` mengganti setiap bagian yang cocok dengan pola. Bagian yang dibuang diganti **spasi**, bukan string kosong, agar dua kata tidak tersambung (misalnya `bagus!mantap` menjadi `bagus mantap`, bukan `bagusmantap`).

Arti simbol regex yang dipakai:

| Simbol | Arti |
|---|---|
| `\d` | satu digit angka 0 sampai 9 |
| `\w` | satu karakter kata (huruf, angka, atau garis bawah) |
| `\s` / `\S` | satu karakter spasi (spasi, tab, baris baru) / satu karakter bukan spasi |
| `+` | satu kali atau lebih dari elemen sebelumnya |
| `*?` | nol kali atau lebih, tetapi sesedikit mungkin (*non-greedy*) |
| `{2,}` | dua kali atau lebih |
| `?` | elemen sebelumnya boleh ada atau tidak (`https?` cocok dengan `http` dan `https`) |
| `\|` | atau |
| `[...]` | salah satu karakter di dalam kurung |
| `[^...]` | karakter apa saja **kecuali** yang ada di dalam kurung |
| `(...)` dan `\1` | kelompok yang ditangkap, dan rujukan ke isi kelompok pertama |
| `\b` | batas kata |

Pola yang dipakai, sesuai urutan eksekusi:

| No | Nama | Pola | Pengganti | Arti | Alasan |
|---|---|---|---|---|---|
| 1 | URL | `https?://\S+\|www\.\S+` | spasi | teks yang diawali `http://`, `https://`, atau `www.` sampai spasi berikutnya | tautan tidak membawa makna keluhan |
| 2 | HTML | `<[^>]+>\|&\w+;` | spasi | tag seperti `<br>` dan entitas seperti `&amp;` | sisa format, bukan isi ulasan |
| 3 | Mention dan hashtag | `[@#]\w+` | spasi | kata yang diawali `@` atau `#` | menyebut akun atau tagar, bukan isi keluhan |
| 4 | Emoji | rentang Unicode emoji (lihat `EMOJI_PATTERN` di 5.5) | spasi | karakter emoji, bendera, simbol bergambar | emoji tidak diproses oleh stemming dan stopword, dan maknanya sudah terwakili oleh kata di sekitarnya |
| 5 | Kata ulang angka | `\b([a-z]{2,})2\b` | `\1` | kata yang diakhiri angka 2 (`fitur2`, `teman2`) | penulisan kata ulang informal; diganti kata dasarnya sebelum angka dihapus, agar tidak tersisa `fitur` lalu angka hilang tanpa jejak |
| 6 | Angka | `\d+` | spasi | satu atau lebih digit berturut-turut | nominal, nomor HP, dan tanggal berbeda-beda di setiap ulasan sehingga tidak membantu klasifikasi |
| 7 | Simbol dan tanda baca | `[^a-z\s]` | spasi | semua karakter selain huruf `a-z` dan spasi | membuang tanda baca, simbol mata uang, dan sisa emoji atau karakter non-Latin |
| 8 | Huruf berulang | `([a-z])\1{2,}` | `\1` | satu huruf yang muncul 3 kali atau lebih berturut-turut | `lemooot` menjadi `lemot`, `parahhh` menjadi `parah` |
| 9 | Spasi berlebih | `\s+` lalu `strip()` | satu spasi | dua atau lebih spasi, tab, atau baris baru | merapikan hasil penggantian sebelumnya |

Catatan perbedaan dengan notebook 00: di notebook 00 huruf berulang dipendekkan menjadi **dua** huruf (`\1\1`) karena hanya dipakai untuk mencocokkan kata kunci pelabelan. Di sini dipendekkan menjadi **satu** huruf (`\1`) karena tujuannya menyatukan varian ejaan menjadi satu token. Akibat sampingnya, kata baku dengan huruf ganda tetap aman karena pola hanya aktif untuk tiga huruf atau lebih (`saat` tidak berubah).

In [ ]:
RE_URL = re.compile(r'https?://\S+|www\.\S+')
RE_HTML = re.compile(r'<[^>]+>|&\w+;')
RE_MENTION = re.compile(r'[@#]\w+')
RE_EMOJI = re.compile(EMOJI_PATTERN)
RE_REDUP = re.compile(r'\b([a-z]{2,})2\b')
RE_DIGIT = re.compile(r'\d+')
RE_NON_ALPHA = re.compile(r'[^a-z\s]')
RE_REPEAT = re.compile(r'([a-z])\1{2,}')
RE_SPACE = re.compile(r'\s+')

REGEX_STEPS = [
    ('URL', RE_URL, ' '),
    ('HTML', RE_HTML, ' '),
    ('Mention dan hashtag', RE_MENTION, ' '),
    ('Emoji', RE_EMOJI, ' '),
    ('Kata ulang angka', RE_REDUP, r'\1'),
    ('Angka', RE_DIGIT, ' '),
    ('Simbol dan tanda baca', RE_NON_ALPHA, ' '),
    ('Huruf berulang', RE_REPEAT, r'\1'),
]


def clean_text_regex(text: str) -> str:
    """Terapkan semua pola regex secara berurutan. Input harus sudah huruf kecil."""
    for _, pattern, repl in REGEX_STEPS:
        text = pattern.sub(repl, text)
    return RE_SPACE.sub(' ', text).strip()

Tabel berikut menguji setiap pola secara terpisah pada contoh kalimat, sehingga efek tiap pola terlihat jelas. Contoh emoji ditulis sebagai escape Unicode (`\U0001F621` adalah wajah marah, `\U0001F44D` adalah jempol).

In [ ]:
regex_examples = {
    'URL': 'cek di https://dana.id/help atau www.dana.id',
    'HTML': 'aplikasi bagus<br>tapi lemot &amp; berat',
    'Mention dan hashtag': '@gopay tolong perbaiki #gopayerror',
    'Emoji': 'saldo hilang \U0001F621\U0001F621 kecewa\U0001F44D',
    'Kata ulang angka': 'fitur2 baru bikin teman2 bingung',
    'Angka': 'nunggu 30 menit saldo 150000 belum masuk',
    'Simbol dan tanda baca': 'error!!! kenapa?? rp.50.000,-',
    'Huruf berulang': 'lemooot parahhh bangettt',
}
regex_demo = pd.DataFrame([
    {'tahap': name, 'pola': pattern.pattern if name != 'Emoji' else 'EMOJI_PATTERN',
     'sebelum': regex_examples[name],
     'sesudah': RE_SPACE.sub(' ', pattern.sub(repl, regex_examples[name])).strip()}
    for name, pattern, repl in REGEX_STEPS
])
regex_demo

Fungsi `clean_text_regex` diterapkan ke kolom `case_folded`. Selain itu, dihitung berapa ulasan yang berubah oleh setiap pola, untuk melihat pola mana yang paling banyak bekerja.

In [ ]:
df['cleaned'] = df['case_folded'].map(clean_text_regex)

regex_impact = pd.DataFrame({
    'ulasan_terdampak': {name: int(df['case_folded'].str.contains(pattern, regex=True).sum())
                         for name, pattern, _ in REGEX_STEPS},
})
regex_impact['persen'] = (regex_impact['ulasan_terdampak'] / len(df) * 100).round(2)
display(regex_impact)

df[['case_folded', 'cleaned']].head(N_EXAMPLES)

### 6.3 Normalisasi slang

Ulasan aplikasi penuh singkatan dan kata tidak baku. Kata yang maknanya sama tetapi ditulis berbeda (`gak`, `ga`, `gk`, `nggak`, `tdk`) akan menjadi fitur yang berbeda jika tidak disatukan. Normalisasi slang memetakan kata tidak baku ke bentuk bakunya menggunakan kamus (dictionary Python) `SLANG`.

Sumber kamus:
- Inti kamus diambil dari kamus `SLANG` di notebook 00, agar pelabelan dan preprocessing memakai pemetaan yang sama.
- Ditambah kata tidak baku yang sering muncul di dataset ini, ditemukan dengan melihat daftar kata terbanyak setelah regex cleaning (misalnya `udh`, `skrg`, `tetep`, `nunggu`, `ilang`).
- Beberapa pemetaan notebook 00 yang khusus untuk pelabelan tidak dipakai di sini. Contohnya `kasih` menjadi `beri` tidak dipakai, karena akan mengubah `terima kasih` menjadi `terima beri`.

Cara kerja: teks dipecah per spasi, setiap kata dicari di kamus, lalu diganti jika ada. Nilai kamus boleh lebih dari satu kata (`gabisa` menjadi `tidak bisa`).

In [ ]:
SLANG = {
    # negasi
    'gak': 'tidak', 'ga': 'tidak', 'gk': 'tidak', 'g': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
    'enggak': 'tidak', 'engga': 'tidak', 'tdk': 'tidak', 'tak': 'tidak', 'kagak': 'tidak', 'kaga': 'tidak',
    'gx': 'tidak', 'ndak': 'tidak', 'kgk': 'tidak', 'gabisa': 'tidak bisa', 'gbs': 'tidak bisa',
    'gakbisa': 'tidak bisa', 'gada': 'tidak ada', 'gaada': 'tidak ada', 'gajelas': 'tidak jelas',
    'blm': 'belum', 'blom': 'belum', 'bkn': 'bukan', 'jgn': 'jangan',
    # kata fungsi dan kata umum
    'bgt': 'banget', 'bngt': 'banget', 'yg': 'yang', 'dgn': 'dengan', 'dg': 'dengan',
    'udh': 'sudah', 'udah': 'sudah', 'sdh': 'sudah', 'dah': 'sudah', 'uda': 'sudah',
    'tlg': 'tolong', 'tolonglah': 'tolong', 'mhn': 'mohon', 'krn': 'karena', 'karna': 'karena',
    'bs': 'bisa', 'bsa': 'bisa', 'dr': 'dari', 'dri': 'dari', 'utk': 'untuk', 'jd': 'jadi', 'jdi': 'jadi',
    'sy': 'saya', 'sya': 'saya', 'aku': 'saya', 'gw': 'saya', 'gue': 'saya', 'gua': 'saya',
    'tp': 'tapi', 'tpi': 'tapi', 'lg': 'lagi', 'lgi': 'lagi', 'sm': 'sama', 'trs': 'terus', 'trus': 'terus',
    'kalo': 'kalau', 'klo': 'kalau', 'klu': 'kalau', 'knp': 'kenapa', 'knpa': 'kenapa', 'napa': 'kenapa',
    'gmn': 'bagaimana', 'gimana': 'bagaimana', 'msh': 'masih', 'jg': 'juga', 'aja': 'saja', 'aj': 'saja',
    'doang': 'saja', 'dlm': 'dalam', 'emg': 'memang', 'emang': 'memang', 'ad': 'ada',
    'skrg': 'sekarang', 'skrng': 'sekarang', 'pdhl': 'padahal', 'cuma': 'hanya', 'cuman': 'hanya',
    'tetep': 'tetap', 'ttp': 'tetap', 'bener': 'benar', 'tau': 'tahu', 'sampe': 'sampai', 'smpe': 'sampai',
    'mulu': 'melulu', 'kayak': 'seperti', 'kek': 'seperti', 'gitu': 'begitu', 'gini': 'begini',
    'org': 'orang', 'bnyk': 'banyak', 'byk': 'banyak', 'dikit': 'sedikit', 'abis': 'habis',
    'liat': 'lihat', 'kpn': 'kapan', 'ko': 'kok', 'donk': 'dong', 'rame': 'ramai',
    'smg': 'semoga', 'smoga': 'semoga', 'moga': 'semoga', 'mudahan': 'semoga',
    'pengin': 'pengen', 'pgn': 'pengen', 'pingin': 'pengen',
    'makasih': 'terima kasih', 'terimakasih': 'terima kasih', 'trimakasih': 'terima kasih',
    'kesel': 'kesal', 'cepet': 'cepat', 'duit': 'uang', 'baget': 'banget', 'bgi': 'bagi', 'smw': 'semua',
    'stlh': 'setelah', 'bbrpa': 'beberapa', 'gmna': 'bagaimana', 'gbisa': 'tidak bisa', 'jlek': 'jelek',
    'tebel': 'tebal', 'ketipu': 'tertipu',
    # kata kerja informal
    'pake': 'pakai', 'dipake': 'dipakai', 'makai': 'memakai', 'digunain': 'digunakan', 'gunain': 'gunakan',
    'nunggu': 'tunggu', 'ngirim': 'kirim', 'masukin': 'masukkan', 'benerin': 'perbaiki',
    'ditambahin': 'ditambahkan', 'tambahin': 'tambahkan', 'tambahain': 'tambahkan', 'tambahkn': 'tambahkan',
    'ditambahkn': 'ditambahkan', 'adain': 'adakan', 'diadain': 'diadakan', 'bikinin': 'buatkan',
    'dibikinin': 'dibuatkan', 'dibuatin': 'dibuatkan', 'buatin': 'buatkan', 'dikasih': 'diberi', 'dikasi': 'diberi',
    'ilang': 'hilang', 'kepotong': 'terpotong', 'betulin': 'perbaiki', 'diupdate': 'update',
    'makenya': 'pakainya', 'make': 'pakai',
    # istilah aplikasi dan teknis
    'apk': 'aplikasi', 'app': 'aplikasi', 'apps': 'aplikasi', 'apl': 'aplikasi', 'aplkasi': 'aplikasi',
    'aplikasih': 'aplikasi', 'apknya': 'aplikasinya',
    'lemod': 'lemot', 'lelet': 'lemot', 'eror': 'error', 'erorr': 'error', 'err': 'error', 'errorr': 'error',
    'ngebug': 'bug', 'ngelag': 'lag', 'ngeleg': 'lag', 'leg': 'lag', 'ngehang': 'hang', 'ngeheng': 'hang',
    'loding': 'loading', 'lodingnya': 'loading', 'loadingnya': 'loading',
    'stak': 'stuck', 'stuk': 'stuck', 'setak': 'stuck', 'darkmode': 'dark mode', 'userfriendly': 'user friendly',
    'pw': 'password', 'tf': 'transfer', 'tranfer': 'transfer', 'trf': 'transfer', 'qiris': 'qris',
    'notif': 'notifikasi', 'hape': 'hp', 'nomer': 'nomor', 'instal': 'install', 'uninstal': 'uninstall',
    'ikln': 'iklan', 'gampang': 'mudah', 'simpel': 'sederhana', 'berfungsj': 'berfungsi',
}


def normalize_slang(text: str) -> str:
    """Ganti setiap kata yang ada di kamus SLANG dengan bentuk bakunya."""
    return ' '.join(SLANG.get(w, w) for w in text.split())


print(f'Jumlah entri kamus SLANG: {len(SLANG)}')
contoh = 'apk nya lemot bgt gk bisa tf udh nunggu lama'
print(contoh, '\nmenjadi:', normalize_slang(contoh))

Kamus diterapkan ke kolom `cleaned`. Tabel di bawahnya menunjukkan kata slang yang paling sering diganti, sekaligus menghitung ciri noise terakhir dari bagian 5.5 (persentase ulasan yang memuat kata slang).

In [ ]:
df['normalized'] = df['cleaned'].map(normalize_slang)

slang_hits = Counter(w for t in df['cleaned'] for w in t.split() if w in SLANG)
n_with_slang = df['cleaned'].map(lambda t: any(w in SLANG for w in t.split())).sum()
print(f'Ulasan yang memuat kata slang : {n_with_slang:,} ({n_with_slang / len(df) * 100:.1f}%)')
print(f'Total kata yang diganti       : {sum(slang_hits.values()):,}')

top_slang = pd.DataFrame(
    [(w, SLANG[w], n) for w, n in slang_hits.most_common(15)],
    columns=['slang', 'bentuk_baku', 'frekuensi'],
)
display(top_slang)
df[['cleaned', 'normalized']].head(N_EXAMPLES)

### 6.4 Tokenisasi

Tokenisasi memecah kalimat menjadi daftar kata (token). Karena tanda baca dan simbol sudah dibuang di tahap regex, pemisahan berdasarkan spasi dengan `str.split()` sudah cukup dan hasilnya sama dengan tokenizer yang lebih kompleks seperti `nltk.word_tokenize`.

Hasil tokenisasi berupa list Python, misalnya `['aplikasi', 'tidak', 'bisa', 'dibuka']`. Bentuk list inilah yang dipakai oleh tahap stopword removal dan stemming.

In [ ]:
df['tokens'] = df['normalized'].str.split()
df['n_tokens'] = df['tokens'].str.len()
print(f'Rata-rata token per ulasan : {df["n_tokens"].mean():.1f}')
df[['normalized', 'tokens']].head(N_EXAMPLES)

### 6.5 Stopword removal

**Stopword** adalah kata yang sangat sering muncul tetapi tidak membawa topik, misalnya `yang`, `dan`, `di`, `saya`. Membuangnya mengecilkan jumlah fitur dan membuat model fokus pada kata yang membedakan label.

Daftar stopword yang dipakai:
1. **Daftar dasar Sastrawi** (`StopWordRemoverFactory().get_stop_words()`).
2. **Tambahan partikel informal** yang sangat sering di ulasan tetapi tidak ada di daftar Sastrawi, seperti `nya`, `sih`, `dong`, `deh`, `kok`, `nih`, `kak`, `min` (sapaan untuk admin).
3. **Dikurangi daftar pengecualian `KEEP_WORDS`**. Kata di daftar ini tetap dipertahankan walaupun ada di daftar Sastrawi, karena penting untuk makna:

| Kata | Alasan dipertahankan |
|---|---|
| `tidak`, `belum`, `bukan`, `jangan`, `kurang` | negasi; tanpa kata ini "tidak bisa dibuka" menjadi "bisa dibuka" yang maknanya terbalik |
| `bisa`, `dapat` | bagian dari keluhan "tidak bisa", "tidak dapat" yang khas Bug Aplikasi |
| `tolong`, `harus`, `ingin`, `seharusnya` | penanda permintaan, khas Permintaan Fitur |
| `lama`, `cepat`, `lambat`, `gagal`, `baru` | penanda performa, bug, dan pembaruan (didaftarkan sebagai pengaman walaupun tidak semuanya ada di daftar Sastrawi) |

Selain itu, token yang hanya terdiri dari **satu huruf** (sisa singkatan seperti `d`, `x`, `e`) ikut dibuang karena tidak bermakna.

In [ ]:
SASTRAWI_STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
EXTRA_STOPWORDS = {
    'nya', 'sih', 'dong', 'deh', 'kok', 'lah', 'kan', 'nih', 'ni', 'tuh', 'loh', 'yah', 'yaa', 'eh', 'mah',
    'si', 'kak', 'min', 'gan', 'ku', 'mu', 'an', 'nge', 'ny',
}
KEEP_WORDS = {
    'tidak', 'belum', 'bukan', 'jangan', 'kurang', 'bisa', 'dapat',
    'tolong', 'harus', 'ingin', 'seharusnya',
    'lama', 'cepat', 'lambat', 'gagal', 'baru',
}
STOPWORDS = (SASTRAWI_STOPWORDS | EXTRA_STOPWORDS) - KEEP_WORDS

print(f'Stopword Sastrawi           : {len(SASTRAWI_STOPWORDS)}')
print(f'Tambahan partikel informal  : {len(EXTRA_STOPWORDS)}')
print(f'Dipertahankan (KEEP_WORDS)  : {sorted(KEEP_WORDS & SASTRAWI_STOPWORDS)}')
print(f'Total stopword yang dibuang : {len(STOPWORDS)}')


def remove_stopwords(tokens: list[str]) -> list[str]:
    """Buang stopword dan token satu huruf."""
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

Fungsi `remove_stopwords` diterapkan ke kolom `tokens`. Tabel di bawahnya menunjukkan stopword yang paling banyak dibuang.

In [ ]:
df['tokens_nostop'] = df['tokens'].map(remove_stopwords)

removed_counts = Counter(t for toks in df['tokens'] for t in toks if t in STOPWORDS or len(t) <= 1)
n_before = df['tokens'].str.len().sum()
n_after = df['tokens_nostop'].str.len().sum()
print(f'Token sebelum : {n_before:,}')
print(f'Token sesudah : {n_after:,} (berkurang {(1 - n_after / n_before) * 100:.1f}%)')

display(pd.DataFrame(removed_counts.most_common(15), columns=['stopword', 'frekuensi_dibuang']))
df[['tokens', 'tokens_nostop']].head(N_EXAMPLES)

### 6.6 Stemming

**Stemming** mengubah kata berimbuhan menjadi kata dasar, misalnya `terpotong` menjadi `potong`, `pembayaran` menjadi `bayar`, `keamanan` menjadi `aman`. Dengan begitu, kata yang berasal dari kata dasar yang sama dihitung sebagai satu fitur.

Library yang dipakai adalah **Sastrawi**, yang menerapkan algoritma Nazief dan Adriani: imbuhan (awalan, akhiran, sisipan, partikel `-lah`/`-kah`, kata ganti `-ku`/`-mu`/`-nya`) dilepas bertahap, lalu hasilnya dicek ke kamus kata dasar Bahasa Indonesia.

**Optimasi dengan cache.** Sastrawi lambat jika dipanggil untuk setiap kata di setiap ulasan (ratusan ribu kali). Karena banyak kata berulang, setiap **kata unik** cukup di-stem satu kali, lalu hasilnya disimpan di dictionary `stem_cache` dan dipakai ulang.

**Proses paralel.** Walaupun memakai cache, Sastrawi masih memerlukan sekitar 10 sampai 60 milidetik untuk setiap kata unik baru. Karena itu, daftar kata unik dibagi ke beberapa proses CPU sekaligus (`multiprocessing`), lalu hasilnya digabung ke `stem_cache`. Hasilnya sama persis dengan stemming satu per satu, hanya lebih cepat.

**Kata yang dikecualikan.** Sastrawi kadang salah memotong kata informal atau istilah teknis, atau menghasilkan kata dasar yang maknanya berbeda jauh. Dua daftar dipakai untuk memperbaikinya:
- `STEM_PROTECT`: kata yang **tidak di-stem**. Contohnya `lemot` (lambat) dipotong menjadi `lot` karena dianggap berawalan `le-`, dan `jaringan` menjadi `jaring`.
- `STEM_OVERRIDE`: kata yang **diberi hasil stem secara manual**. Contohnya `perbaiki`, `diperbaiki`, dan `perbaikan` oleh Sastrawi menjadi `baik`, sehingga keluhan "tolong diperbaiki" tidak bisa dibedakan dari pujian "aplikasinya baik". Ketiganya dipetakan ke `perbaiki`.

Kedua daftar disusun dari pemeriksaan hasil stemming kata yang paling sering muncul (lihat tabel di bawah).

In [ ]:
stemmer = StemmerFactory().create_stemmer()

STEM_PROTECT = {
    'lemot', 'error', 'bug', 'crash', 'loading', 'lag', 'hang', 'stuck', 'update', 'login', 'logout',
    'otp', 'pin', 'ui', 'ux', 'qris', 'paylater', 'topup', 'cashback', 'voucher', 'promo', 'install',
    'uninstall', 'download', 'server', 'email', 'password', 'scan', 'refund', 'pending', 'upgrade',
    'jaringan',
}
STEM_OVERRIDE = {
    'perbaiki': 'perbaiki', 'diperbaiki': 'perbaiki', 'perbaikan': 'perbaiki', 'memperbaiki': 'perbaiki',
    'perbaikannya': 'perbaiki', 'diperbaikin': 'perbaiki',
}


def stem_word(word: str) -> str:
    """Stem satu kata: pakai STEM_OVERRIDE, lewati STEM_PROTECT, selain itu Sastrawi."""
    if word in STEM_OVERRIDE:
        return STEM_OVERRIDE[word]
    if word in STEM_PROTECT:
        return word
    return stemmer.stem(word)


unique_words = sorted({t for toks in df['tokens_nostop'] for t in toks})
n_proc = os.cpu_count() or 1
start = time.time()
with get_context('fork').Pool(n_proc) as pool:
    stem_cache: dict[str, str] = dict(zip(unique_words, pool.map(stem_word, unique_words, chunksize=200)))
elapsed = time.time() - start

df['tokens_stemmed'] = df['tokens_nostop'].map(lambda toks: [stem_cache[t] for t in toks])
print(f'Kata unik yang di-stem : {len(stem_cache):,}')
print(f'Jumlah proses CPU      : {n_proc}')
print(f'Waktu stemming         : {elapsed:.1f} detik')
df[['tokens_nostop', 'tokens_stemmed']].head(N_EXAMPLES)

Tabel berikut menunjukkan kata yang paling sering berubah oleh stemming. Tabel ini dipakai untuk memeriksa kualitas Sastrawi. Hasil yang kurang tepat dicatat sebagai bahan diskusi di laporan, misalnya kata yang bermakna berbeda setelah di-stem (`pengguna` menjadi `guna`, `pembaruan` menjadi `baru`). Kata yang hasilnya merusak makna secara berat sudah dimasukkan ke `STEM_PROTECT` atau `STEM_OVERRIDE`.

In [ ]:
token_freq = Counter(t for toks in df['tokens_nostop'] for t in toks)
changed = pd.DataFrame(
    [(w, stem_cache[w], n) for w, n in token_freq.most_common() if stem_cache[w] != w][:25],
    columns=['kata', 'hasil_stem', 'frekuensi'],
)
changed

### 6.7 Menggabungkan token menjadi `clean_text`

Token hasil stemming digabung kembali dengan spasi menjadi `clean_text`. Kolom inilah yang dipakai oleh BoW dan TF-IDF di notebook 02 dan Word2Vec di notebook 04.

Ulasan yang `clean_text`-nya **kosong** (semua katanya terbuang karena hanya berisi angka, emoji, atau stopword) dibuang, karena tidak memiliki fitur sama sekali.

In [ ]:
df['clean_text'] = df['tokens_stemmed'].str.join(' ')
df['n_words_clean'] = df['tokens_stemmed'].str.len()

empty_mask = df['clean_text'].str.strip() == ''
n_empty = int(empty_mask.sum())
print(f'Ulasan dengan clean_text kosong : {n_empty}')
if n_empty:
    display(df.loc[empty_mask, ['text', 'label']].head(10))

df = df[~empty_mask].reset_index(drop=True)
print(f'Jumlah ulasan akhir             : {len(df):,}')

## 7. Ringkasan preprocessing

### 7.1 Tabel before/after per tahap

Lima contoh ulasan dipilih secara acak (dengan `random_state` tetap) dari ulasan yang memiliki noise (angka, emoji, tanda baca berulang, huruf berulang, atau kata slang) dan panjangnya 8 sampai 30 kata, agar efek setiap tahap terlihat. Satu ulasan diambil dari setiap label.

In [ ]:
has_noise = (df['text'].str.contains(r'\d|[!?]{2,}|([A-Za-z])\1{2,}', regex=True)
             | df['text'].str.contains(EMOJI_PATTERN, regex=True)
             | df['cleaned'].map(lambda t: any(w in SLANG for w in t.split())))
pool = df[has_noise & df['n_words_raw'].between(8, 30)]
examples = (pool.groupby('label', group_keys=False)
                .sample(n=1, random_state=RANDOM_STATE)
                .set_index('label').reindex(LABELS).reset_index())

STEP_COLUMNS = ['text', 'case_folded', 'cleaned', 'normalized', 'clean_text']
before_after = examples[['label'] + STEP_COLUMNS].head(N_EXAMPLES)
before_after

Tabel yang sama dalam bentuk **vertikal** (satu ulasan per blok), lebih mudah dibaca dan disalin ke laporan.

In [ ]:
before_after_long = (before_after
                     .assign(contoh=range(1, len(before_after) + 1))
                     .melt(id_vars=['contoh', 'label'], value_vars=STEP_COLUMNS, var_name='tahap', value_name='hasil'))
before_after_long['tahap'] = pd.Categorical(before_after_long['tahap'], categories=STEP_COLUMNS, ordered=True)
before_after_long = before_after_long.sort_values(['contoh', 'tahap']).set_index(['contoh', 'label', 'tahap'])
before_after_long

### 7.2 Tabel token

Tabel token memperlihatkan daftar token untuk contoh yang sama pada tiga tahap: setelah tokenisasi, setelah stopword removal, dan setelah stemming, beserta jumlah token di setiap tahap.

In [ ]:
token_table = pd.DataFrame({
    'label': examples['label'],
    'tokens': examples['tokens'],
    'tokens_nostop': examples['tokens_nostop'],
    'tokens_stemmed': examples['tokens_stemmed'],
    'jumlah_token': examples['tokens'].str.len().astype(str) + ' / '
                    + examples['tokens_nostop'].str.len().astype(str) + ' / '
                    + examples['tokens_stemmed'].str.len().astype(str),
}).head(N_EXAMPLES)
token_table

### 7.3 Statistik sebelum dan sesudah preprocessing

Ukuran kosakata (jumlah kata unik) menentukan jumlah fitur BoW dan TF-IDF di notebook 02. Penurunan kosakata menunjukkan seberapa banyak varian ejaan, slang, dan imbuhan yang berhasil disatukan.

In [ ]:
def vocab_size(series_of_token_lists: pd.Series) -> int:
    """Jumlah kata unik di seluruh ulasan."""
    return len({t for toks in series_of_token_lists for t in toks})

stats = pd.DataFrame([
    ('Teks asli (huruf kecil, split spasi)', vocab_size(df['case_folded'].str.split()), df['case_folded'].str.split().str.len().mean()),
    ('Setelah regex cleaning', vocab_size(df['cleaned'].str.split()), df['cleaned'].str.split().str.len().mean()),
    ('Setelah normalisasi slang', vocab_size(df['tokens']), df['tokens'].str.len().mean()),
    ('Setelah stopword removal', vocab_size(df['tokens_nostop']), df['tokens_nostop'].str.len().mean()),
    ('Setelah stemming (clean_text)', vocab_size(df['tokens_stemmed']), df['tokens_stemmed'].str.len().mean()),
], columns=['tahap', 'jumlah_kata_unik', 'rata_rata_kata_per_ulasan']).set_index('tahap')
stats['rata_rata_kata_per_ulasan'] = stats['rata_rata_kata_per_ulasan'].round(2)
display(stats)

dup_clean = df['clean_text'].duplicated().sum()
print(f'Ulasan dibuang karena hampir kembar : {n_removed}')
print(f'Ulasan dibuang karena clean_text kosong : {n_empty}')
print(f'clean_text yang sama dengan ulasan lain : {dup_clean:,} '
      '(dipertahankan: teks aslinya berbeda, umumnya ulasan pendek seperti "aplikasi lemot")')

Grafik berikut membandingkan sebaran panjang ulasan sebelum dan sesudah preprocessing. Sumbu horizontal dibatasi sampai persentil 99 agar bentuk sebaran terlihat jelas.

In [ ]:
x_max = int(df['n_words_raw'].quantile(0.99))
bins = np.arange(0, x_max + 2, 2)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['n_words_raw'].clip(upper=x_max), bins=bins, alpha=0.6, label='Sebelum (text)', color='#7F7F7F')
ax.hist(df['n_words_clean'].clip(upper=x_max), bins=bins, alpha=0.6, label='Sesudah (clean_text)', color='#0072B2')
ax.set_title('Panjang ulasan sebelum dan sesudah preprocessing')
ax.set_xlabel('Jumlah kata per ulasan')
ax.set_ylabel('Jumlah ulasan')
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, '06_length_before_after')
plt.show()

## 8. EDA setelah preprocessing

### 8.1 Kata terbanyak per label

Setelah stopword dibuang dan kata disatukan, kata terbanyak per label mulai memperlihatkan ciri khas setiap label, misalnya `error`, `masuk`, `perbaiki` untuk Bug Aplikasi, `lemot`, `lama`, `loading` untuk Performa Aplikasi, dan `fitur`, `tambah` untuk Permintaan Fitur. Kata yang muncul di semua label (seperti `aplikasi`) tetap terlihat; bobotnya nanti diturunkan oleh TF-IDF di notebook 02.

In [ ]:
top_words = {}
for lab in LABELS:
    counts = Counter(t for toks in df.loc[df['label'] == lab, 'tokens_stemmed'] for t in toks)
    top_words[lab] = counts

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, lab in zip(axes.flat, LABELS):
    words, freqs = zip(*top_words[lab].most_common(TOP_N_WORDS))
    ax.barh(words[::-1], freqs[::-1], color=LABEL_COLORS[lab])
    ax.set_title(lab)
    ax.set_xlabel('Frekuensi')
axes.flat[-1].axis('off')
fig.suptitle(f'{TOP_N_WORDS} kata terbanyak per label setelah preprocessing', fontsize=14)
plt.tight_layout()
save_fig(fig, '07_top_words_per_label')
plt.show()

top_words_table = pd.DataFrame({lab: [f'{w} ({n})' for w, n in top_words[lab].most_common(10)] for lab in LABELS},
                               index=range(1, 11))
top_words_table

### 8.2 Word cloud per label

Word cloud menampilkan kata dengan ukuran sebanding frekuensinya. Word cloud dibuat dari frekuensi kata di `clean_text` setiap label (`generate_from_frequencies`), dengan warna sesuai `LABEL_COLORS` dan `random_state` tetap agar tata letaknya selalu sama.

Grafik 8.1 memperlihatkan bahwa `tidak`, `bisa`, `aplikasi`, dan nama aplikasi (`dana`, `ovo`, `gopay`) berada di urutan atas pada hampir semua label, sehingga word cloud akan didominasi kata yang sama. Karena itu, khusus untuk word cloud, kata di `WORDCLOUD_EXCLUDE` tidak ditampilkan agar kata khas setiap label lebih terlihat. Pengecualian ini **hanya untuk visualisasi**; `clean_text` tidak berubah dan kata-kata tersebut tetap dipakai model.

In [ ]:
def single_color(hex_color: str):
    """Fungsi pewarna word cloud: semua kata memakai satu warna label."""
    return lambda *args, **kwargs: hex_color

WORDCLOUD_EXCLUDE = {'tidak', 'bisa', 'aplikasi', 'dana', 'ovo', 'gopay', 'shopeepay', 'shopee'}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, lab in zip(axes.flat, LABELS):
    freqs = {w: n for w, n in top_words[lab].most_common(200) if w not in WORDCLOUD_EXCLUDE}
    wc = WordCloud(width=800, height=450, background_color='white', max_words=80,
                   random_state=RANDOM_STATE, color_func=single_color(LABEL_COLORS[lab]))
    wc.generate_from_frequencies(freqs)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(lab, fontsize=13)
    ax.axis('off')
axes.flat[-1].axis('off')
fig.suptitle('Word cloud per label (clean_text, tanpa kata yang umum di semua label)', fontsize=15)
plt.tight_layout()
save_fig(fig, '08_wordcloud_per_label')
plt.show()

## 9. Menyimpan output

File yang disimpan di `/kaggle/working`:

| File | Isi |
|---|---|
| `clean_dataset_part1.csv` | Dataset hasil preprocessing, kolom `text, clean_text, label, app, rating, source` |
| `raw_dataset.csv` | Salinan dataset asli dari Input (untuk folder pengumpulan) |
| `preprocessing_examples.csv` | Tabel before/after lima contoh ulasan untuk laporan |
| `preprocessing_stats.csv` | Statistik kosakata dan panjang ulasan per tahap |
| `figures/*.png` | Grafik EDA untuk laporan dan slide (200 dpi) |

Kolom wajib tugas adalah `text`, `clean_text`, dan `label`. Kolom `app`, `rating`, dan `source` disertakan sebagai kolom bantu untuk analisis kesalahan di notebook 03. Kolom tahap antara (`cleaned`, `tokens`, dan seterusnya) tidak disimpan karena dapat dibuat ulang dari notebook ini.

In [ ]:
OUTPUT_COLUMNS = ['text', 'clean_text', 'label', 'app', 'rating', 'source']
df_out = df[OUTPUT_COLUMNS]

assert df_out['clean_text'].str.strip().ne('').all(), 'Masih ada clean_text kosong.'
assert set(df_out['label']) <= set(LABELS), 'Ada label di luar LABELS.'

df_out.to_csv(f'{OUTPUT_DIR}/{CLEAN_FILE}', index=False)
before_after.to_csv(f'{OUTPUT_DIR}/preprocessing_examples.csv', index=False)
stats.to_csv(f'{OUTPUT_DIR}/preprocessing_stats.csv')

final_counts = df_out['label'].value_counts().reindex(LABELS)
display(pd.DataFrame({'jumlah': final_counts, 'persen': (final_counts / len(df_out) * 100).round(2)}))
print(f'Tersimpan: {OUTPUT_DIR}/{CLEAN_FILE} ({len(df_out):,} baris, {df_out.shape[1]} kolom)')
print('Isi OUTPUT_DIR:', sorted(os.listdir(OUTPUT_DIR)))
df_out.sample(5, random_state=RANDOM_STATE)

## 10. Penutup

Ringkasan notebook ini:
1. Dataset dibaca dari `raw_dataset.csv`, ulasan hampir kembar dibuang, lalu dianalisis lewat EDA (distribusi label, aplikasi, rating, panjang ulasan, ciri noise).
2. Teks dibersihkan bertahap: case folding, regex cleaning (URL, HTML, mention, emoji, angka, simbol, huruf berulang), normalisasi slang, tokenisasi, stopword removal dengan negasi dipertahankan, dan stemming Sastrawi dengan cache.
3. Hasilnya disimpan sebagai `clean_dataset_part1.csv`.

Catatan penting untuk tahap berikutnya:
- Label tidak seimbang (lihat 5.1), sehingga notebook 03 memakai `class_weight='balanced'` dan F1 macro.
- Kata yang muncul di semua label (misalnya `aplikasi`, `saldo`) tetap ada di `clean_text`; TF-IDF di notebook 02 akan menurunkan bobotnya.

**Notebook berikutnya:** `02_feature_extraction.ipynb` (BoW dan TF-IDF, nama Kaggle `02-text-representation-app-feedback`), dengan Input Output notebook ini (`clean_dataset_part1.csv`).